In [8]:
import os
import cv2
import shutil
import zipfile


In [5]:
input_folders = [
    '../data/preprocessed/divided',
    '../data/original/single_floors'
]

output_folder = '../data/preprocessed/resized'

# Remove all previous files in directory
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
# Make an output directory
os.makedirs(output_folder, exist_ok=True)


In [6]:
target_size = 1024

def rotate_and_resize(image):
    """
    Rotate the image to make the longest side vertical and resize it so that the longest side is equal to `target_size`.
    """
    height, width = image.shape[:2]
    
    # Rotate the image if the width is greater than the height
    if width > height:
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        height, width = width, height  # Swap height and width after rotation
    
    # Calculate the scaling ratio
    scale_ratio = target_size / max(height, width)
    
    # Calculate new dimensions
    new_width = int(width * scale_ratio)
    new_height = int(height * scale_ratio)
    
    # Ensure the longest side is exactly `target_size`
    if new_height > new_width:
        new_height = target_size
        new_width = int(width * (target_size / height))
    else:
        new_width = target_size
        new_height = int(height * (target_size / width))

    resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
    
    return resized_image

In [7]:
for input_folder in input_folders:
    for filename in sorted(os.listdir(input_folder)):
        if filename.endswith('.png') or filename.endswith('.jpg') or filename.endswith('.webp'):
            file_path = os.path.join(input_folder, filename)
            output_file_path = os.path.join(output_folder, filename)
            
            # Read the image
            image = cv2.imread(file_path)
            if image is None:
                print(f"Warning: Could not read image {file_path}. Skipping.")
                continue
            
            # Rotate and resize the image
            processed_image = rotate_and_resize(image)
            
            # Save the processed image
            cv2.imwrite(output_file_path, processed_image)
            print(f"Processed and saved: {output_file_path}")

Processed and saved: ../data/preprocessed/resized/00001_0.png
Processed and saved: ../data/preprocessed/resized/00001_1.png
Processed and saved: ../data/preprocessed/resized/00001_2.png
Processed and saved: ../data/preprocessed/resized/00001_3.png
Processed and saved: ../data/preprocessed/resized/00001_4.png
Processed and saved: ../data/preprocessed/resized/00002_0.png
Processed and saved: ../data/preprocessed/resized/00002_1.png
Processed and saved: ../data/preprocessed/resized/00002_2.png
Processed and saved: ../data/preprocessed/resized/00002_3.png
Processed and saved: ../data/preprocessed/resized/00002_4.png
Processed and saved: ../data/preprocessed/resized/00003_0.png
Processed and saved: ../data/preprocessed/resized/00003_1.png
Processed and saved: ../data/preprocessed/resized/00003_2.png
Processed and saved: ../data/preprocessed/resized/00003_3.png
Processed and saved: ../data/preprocessed/resized/00003_4.png
Processed and saved: ../data/preprocessed/resized/00004_0.png
Processe

In [9]:
# Create a zip file of all images in the output folder
zip_file_path = '../data/preprocessed/resized.zip'
with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_folder):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_folder)
            zipf.write(file_path, arcname)